# How to create a simulation

This tutorial demonstrates how to create a simulation.

## What is a simulation?

A simulation contains selected sensors, sources to model ray-trace in space.

## Prerequisites

### Perform imports

In [1]:
from pathlib import Path

from ansys.speos.core import Project, Speos, launcher
from ansys.speos.core.generic.version_checker import server_version_checker
from ansys.speos.core.kernel.client import (
    default_docker_channel,
)
from ansys.speos.core.simulation import (
    SimulationInteractive,
    SimulationInverse,
    SimulationVirtualBSDF,
)


### Define constants

The constants help ensure consistency and avoid repetition throughout the example.

In [2]:
HOSTNAME = "localhost"
GRPC_PORT = 50098  # Be sure the Speos GRPC Server has been started on this port.
USE_DOCKER = True  # Set to False if you're running this example locally as a Notebook.
SOURCE_NAME = "Surface.1"
SENSOR_NAME = "Irradiance.1"

## Model Setup

### Load assets
The assets used to run this example are available in the
[PySpeos repository](https://github.com/ansys/pyspeos/) on GitHub.

> **Note:** Make sure you
> have downloaded simulation assets and set ``assets_data_path``
> to point to the assets folder.

In [3]:
if USE_DOCKER:  # Running on the remote server.
    assets_data_path = Path("/app") / "assets"
else:
    assets_data_path = Path("/path/to/your/download/assets/directory")

### Connect to the RPC Server
This Python client connects to a server where the Speos engine
is running as a service. In this example, the server and
client are the same machine. The launch_local_speos_rpc_method can
be used to start a local instance of the service..

In [4]:
if USE_DOCKER:
    speos = Speos(channel=default_docker_channel())
else:
    speos = launcher.launch_local_speos_rpc_server(port=GRPC_PORT)

/home/runner/work/pyspeos/pyspeos/.venv/lib/python3.15/site-packages/ansys/tools/common/cyberchannel.py:201: UserWarning: Starting gRPC client without TLS on localhost:50098. This is INSECURE. Consider using a secure connection.
  warn(f"Starting gRPC client without TLS on {target}. This is INSECURE. Consider using a secure connection.")


### Create a new project

The only way to create a simulation using the core layer, is to create it from a
project. The ``Project`` class is instantiated by passing a ``Speos`` instance.

In [5]:
p = Project(speos=speos)
print(p)

{
    "name": "",
    "description": "",
    "metadata": {},
    "part_guid": "",
    "sub_scene_anchor_axis_system": [],
    "sources": [],
    "sensors": [],
    "simulations": [],
    "materials": [],
    "scenes": [],
    "textures_3d": []
}


### Prepare prerequisites

Create the necessary elements for a simulation: Sensor, source, root part, optical property are
prerequisites.

### Prepare the root part

In [6]:
root_part = p.create_root_part()
body_1 = root_part.create_body(name="Body.1")
face_1 = body_1.create_face(name="Face.1")
face_1.vertices = [0, 1, 2, 0, 2, 2, 1, 2, 2]
face_1.facets = [0, 1, 2]
face_1.normals = [0, 0, 1, 0, 0, 1, 0, 0, 1]
root_part.commit()

### Prepare an optical property
Create Optical Property

In [7]:
opt_prop = p.create_optical_property("Material.1")
opt_prop.set_volume_opaque().set_surface_mirror()

Choose the geometry for this optical property : Body.1

In [8]:
opt_prop.geometries = [body_1]
opt_prop.commit()

### Prepare an irradiance sensor

In [9]:
sensor1 = p.create_sensor(name=SENSOR_NAME)
# set type to colorimetric or spectral so that the sensor can be used both in
# direct and inverse simulation
sensor1.set_type_colorimetric()
sensor1.commit()

### Prepare a surface source

In [10]:
source1 = p.create_source(name=SOURCE_NAME)
source1.set_exitance_constant().geometries = [(face_1, True)]
# define a spectrum which is not monochromatic so it can be used in both direct and inverse
# simulation
source1.spectrum.set_blackbody()
source1.commit()

## Create a simulation

In [11]:
simulation1 = p.create_simulation(name="Simulation.1")
simulation1.sensor_paths = [sensor1]  # use sensor instance object
simulation1.source_paths = [source1]  # use source instance object
print(simulation1)
simulation1.commit()
print(simulation1)

local: {
    "name": "Simulation.1",
    "sensor_paths": [
        "Irradiance.1"
    ],
    "source_paths": [
        "Surface.1"
    ],
    "geometries": {
        "geo_paths": []
    },
    "display_name": "",
    "description": "",
    "metadata": {},
    "simulation_guid": "",
    "source_groups": [],
    "simulation": {
        "direct_mc_simulation_template": {
            "geom_distance_tolerance": 0.01,
            "max_impact": 100,
            "weight": {
                "minimum_energy_percentage": 0.005
            },
            "dispersion": true,
            "colorimetric_standard": "CIE_1931",
            "fast_transmission_gathering": false,
            "ambient_material_uri": "",
            "stop_condition_rays_number": "200000",
            "automatic_save_frequency": 1800
        },
        "name": "Simulation.1",
        "description": "",
        "metadata": {},
        "scene_guid": "50bb78db-9822-412b-9b80-e568f909f091",
        "simulation_path": "Simulation.

/home/runner/work/pyspeos/pyspeos/src/ansys/speos/core/simulation.py:1379: UserWarning: Please note that setting a value for light expert option forces a sensorcommit when committing the Simulation class
  self.light_expert = default_parameters.light_expert


### Set simulation characteristics

Simulation is defined with the same default values as the GUI speos.

If the user would like to modify the simulation characteristics,
it is possible to do so by setting the simulation characteristics as below.

In [12]:
simulation2_direct = p.create_simulation(name="Simulation.2")
simulation2_direct.ambient_material_file_uri = assets_data_path / "AIR.material"
simulation2_direct.set_colorimetric_standard_CIE_1964().set_weight_none().set_dispersion = False
simulation2_direct.geom_distance_tolerance = 0.01
simulation2_direct.max_impact = 200
simulation2_direct.sensor_paths = [SENSOR_NAME]  # use sensor instance name
simulation2_direct.source_paths = [SOURCE_NAME]  # use source instance name
simulation2_direct.commit()
print(simulation2_direct)

{
    "name": "Simulation.2",
    "metadata": {
        "UniqueId": "35cdb0a7-a4ae-47dd-bfa5-2c3a1d8a8667"
    },
    "simulation_guid": "ac564f01-9223-456a-8f0c-a8fc9a19c528",
    "sensor_paths": [
        "Irradiance.1"
    ],
    "source_paths": [
        "Surface.1"
    ],
    "geometries": {
        "geo_paths": []
    },
    "display_name": "",
    "description": "",
    "source_groups": [],
    "simulation": {
        "direct_mc_simulation_template": {
            "geom_distance_tolerance": 0.01,
            "max_impact": 200,
            "colorimetric_standard": "CIE_1964",
            "dispersion": true,
            "ambient_material_uri": "/app/assets/AIR.material",
            "fast_transmission_gathering": false,
            "stop_condition_rays_number": "200000",
            "automatic_save_frequency": 1800
        },
        "name": "Simulation.2",
        "description": "",
        "metadata": {},
        "scene_guid": "50bb78db-9822-412b-9b80-e568f909f091",
        "sim

### Fast transmission gathering

Fast transmission gathering accelerates the simulation by neglecting the light refraction that
occurs when the light is transmitted through a transparent surface.

Starting with Speos 2027 R1, the geometries on which it is applied are selected with the
project feature created by ``create_fast_transmission_gathering()``, while the boolean
``fast_transmission_gathering`` property of the simulation activates or deactivates it.

Geometries can be given as ``Body``, ``GeoRef`` or geo-path strings.

> **Note:** the geometries are stored at scene level, therefore they are shared by all the
> simulations of the project, and they are sent to the server at commit.

In [13]:
if server_version_checker.is_version_supported(2027, 1, 0):
    print(simulation2_direct.fast_transmission_gathering)  # False: deactivated by default

    ftg = p.create_fast_transmission_gathering()
    ftg.geometries = [body_1]
    ftg.commit()
    simulation2_direct.fast_transmission_gathering = True
    simulation2_direct.commit()
    print(simulation2_direct.fast_transmission_gathering)

False
True


Deactivate it: this will not delete the geometries, but will deactivate the feature for
this simulation.

In [14]:
if server_version_checker.is_version_supported(2027, 1, 0):
    simulation2_direct.fast_transmission_gathering = False
    simulation2_direct.commit()
    print(simulation2_direct.fast_transmission_gathering)

False


### Read information

Read simulation information

In [15]:
print(simulation1)

{
    "name": "Simulation.1",
    "metadata": {
        "UniqueId": "30c6740d-912a-4a0f-9708-a7836b51f013"
    },
    "simulation_guid": "e430141b-b98f-4501-b140-fb09f410cfe9",
    "sensor_paths": [
        "Irradiance.1"
    ],
    "source_paths": [
        "Surface.1"
    ],
    "geometries": {
        "geo_paths": []
    },
    "display_name": "",
    "description": "",
    "source_groups": [],
    "simulation": {
        "direct_mc_simulation_template": {
            "geom_distance_tolerance": 0.01,
            "max_impact": 100,
            "weight": {
                "minimum_energy_percentage": 0.005
            },
            "dispersion": true,
            "colorimetric_standard": "CIE_1931",
            "fast_transmission_gathering": false,
            "ambient_material_uri": "",
            "stop_condition_rays_number": "200000",
            "automatic_save_frequency": 1800
        },
        "name": "Simulation.1",
        "description": "",
        "metadata": {},
        

Read project information

In [16]:
print(p)

{
    "part_guid": "1e0eff81-cb7c-46e0-93a3-45bb3d37924b",
    "sources": [
        {
            "name": "Surface.1",
            "metadata": {
                "UniqueId": "c614c2f3-13cd-48ee-a836-a615bca683de"
            },
            "source_guid": "e6b40c3d-29ff-429e-b3c4-604c0adc97bf",
            "display_name": "",
            "description": "",
            "source": {
                "name": "Surface.1",
                "surface": {
                    "luminous_flux": {
                        "luminous_value": 683.0
                    },
                    "intensity_guid": "ec427074-2e2a-4145-b589-9ecbd5228dfc",
                    "exitance_constant": {
                        "geo_paths": [
                            {
                                "geo_path": "Body.1/Face.1",
                                "reverse_normal": true
                            }
                        ]
                    },
                    "spectrum_guid": "bcbf92bc-9b3f-4e47-9

## Update simulation settings

If you are manipulating a simulation already committed, remember to commit your changes.

If you don't, you will still only watch what is committed on the server.

In [17]:
simulation1.ambient_material_file_uri = assets_data_path / "AIR.material"
simulation1.commit()
print(simulation1)

{
    "name": "Simulation.1",
    "metadata": {
        "UniqueId": "30c6740d-912a-4a0f-9708-a7836b51f013"
    },
    "simulation_guid": "e430141b-b98f-4501-b140-fb09f410cfe9",
    "sensor_paths": [
        "Irradiance.1"
    ],
    "source_paths": [
        "Surface.1"
    ],
    "geometries": {
        "geo_paths": []
    },
    "display_name": "",
    "description": "",
    "source_groups": [],
    "simulation": {
        "direct_mc_simulation_template": {
            "geom_distance_tolerance": 0.01,
            "max_impact": 100,
            "weight": {
                "minimum_energy_percentage": 0.005
            },
            "dispersion": true,
            "ambient_material_uri": "/app/assets/AIR.material",
            "colorimetric_standard": "CIE_1931",
            "fast_transmission_gathering": false,
            "stop_condition_rays_number": "200000",
            "automatic_save_frequency": 1800
        },
        "name": "Simulation.1",
        "description": "",
        

## Reset

Possibility to reset local values from the one available in the server.

In [18]:
simulation1.max_impact = 1000  # adjust max impact but no commit
simulation1.reset()  # reset -> this will apply the server value to the local value
simulation1.delete()  # delete (to display the local value with the below print)
print(simulation1)

local: {
    "name": "Simulation.1",
    "sensor_paths": [
        "Irradiance.1"
    ],
    "source_paths": [
        "Surface.1"
    ],
    "geometries": {
        "geo_paths": []
    },
    "display_name": "",
    "description": "",
    "metadata": {},
    "simulation_guid": "",
    "source_groups": [],
    "simulation": {
        "direct_mc_simulation_template": {
            "geom_distance_tolerance": 0.01,
            "max_impact": 100,
            "weight": {
                "minimum_energy_percentage": 0.005
            },
            "dispersion": true,
            "ambient_material_uri": "/app/assets/AIR.material",
            "colorimetric_standard": "CIE_1931",
            "fast_transmission_gathering": false,
            "stop_condition_rays_number": "200000",
            "automatic_save_frequency": 1800
        },
        "name": "Simulation.1",
        "description": "",
        "metadata": {},
        "scene_guid": "50bb78db-9822-412b-9b80-e568f909f091",
        "simula

## Other simulation examples

### Inverse simulation

In [19]:
simulation3 = p.create_simulation(name="Simulation.3", feature_type=SimulationInverse)
simulation3.sensor_paths = [SENSOR_NAME]
simulation3.source_paths = [source1]
simulation3.commit()
print(simulation3)

{
    "name": "Simulation.3",
    "metadata": {
        "UniqueId": "42d66414-db98-43d1-92a4-c199fe17a015"
    },
    "simulation_guid": "dc016c71-fbda-40c8-9b45-42127f7cd34f",
    "sensor_paths": [
        "Irradiance.1"
    ],
    "source_paths": [
        "Surface.1"
    ],
    "geometries": {
        "geo_paths": []
    },
    "display_name": "",
    "description": "",
    "source_groups": [],
    "simulation": {
        "inverse_mc_simulation_template": {
            "geom_distance_tolerance": 0.01,
            "max_impact": 100,
            "weight": {
                "minimum_energy_percentage": 0.005
            },
            "number_of_gathering_rays_per_source": 1,
            "colorimetric_standard": "CIE_1931",
            "dispersion": false,
            "splitting": false,
            "maximum_gathering_error": 0,
            "maximum_gathering_error_percentage": 0.0,
            "fast_transmission_gathering": false,
            "ambient_material_uri": "",
            "o

#### Optimized propagation

By default the inverse simulation sends the same number of passes for each pixel of the image.
The number of passes is driven by the ``stop_condition_passes_number`` property.

In [20]:
print(simulation3.stop_condition_passes_number)
simulation3.stop_condition_passes_number = 10
print(simulation3.stop_condition_passes_number)

5
10


Starting with Speos 2026 R1 SP3, the optimized propagation algorithm adapts the number of passes
per pixel to send the optimal number of rays according to the signal each pixel needs.

The relative mode stops a pixel once its standard deviation drops below a given percentage.

In [21]:
relative = simulation3.set_optimized_propagation_relative()
relative.min_pass_number = 200
relative.relative_value = 5
print(relative)

The absolute mode stops a pixel once its standard deviation drops below an absolute
photometric value.

In [22]:
absolute = simulation3.set_optimized_propagation_absolute()
absolute.min_pass_number = 200
absolute.absolute_value = 3
print(absolute)

Switch back to a fixed number of passes by setting the stop condition on the passes number.

In [23]:
simulation3.stop_condition_passes_number = 5
simulation3.commit()
print(simulation3)

{
    "name": "Simulation.3",
    "metadata": {
        "UniqueId": "42d66414-db98-43d1-92a4-c199fe17a015"
    },
    "simulation_guid": "dc016c71-fbda-40c8-9b45-42127f7cd34f",
    "sensor_paths": [
        "Irradiance.1"
    ],
    "source_paths": [
        "Surface.1"
    ],
    "geometries": {
        "geo_paths": []
    },
    "display_name": "",
    "description": "",
    "source_groups": [],
    "simulation": {
        "inverse_mc_simulation_template": {
            "geom_distance_tolerance": 0.01,
            "max_impact": 100,
            "weight": {
                "minimum_energy_percentage": 0.005
            },
            "number_of_gathering_rays_per_source": 1,
            "colorimetric_standard": "CIE_1931",
            "dispersion": false,
            "splitting": false,
            "maximum_gathering_error": 0,
            "maximum_gathering_error_percentage": 0.0,
            "fast_transmission_gathering": false,
            "ambient_material_uri": "",
            "o

### Interactive simulation

In [24]:
simulation4 = p.create_simulation(name="Simulation.4", feature_type=SimulationInteractive)
simulation4.source_paths = [source1]
simulation4.commit()
print(simulation4)

{
    "name": "Simulation.4",
    "metadata": {
        "UniqueId": "4b05c18d-b3c7-4c29-9d11-1f6f5de5115a"
    },
    "simulation_guid": "63f0d56e-7b6c-4eff-8d91-9af648f2f910",
    "source_paths": [
        "Surface.1"
    ],
    "geometries": {
        "geo_paths": []
    },
    "display_name": "",
    "description": "",
    "sensor_paths": [],
    "source_groups": [],
    "simulation": {
        "name": "Simulation.4",
        "interactive_simulation_template": {
            "geom_distance_tolerance": 0.01,
            "max_impact": 100,
            "weight": {
                "minimum_energy_percentage": 0.005
            },
            "colorimetric_standard": "CIE_1931",
            "ambient_material_uri": "",
            "rays_number_per_sources": [],
            "light_expert": false,
            "impact_report": false
        },
        "description": "",
        "metadata": {},
        "scene_guid": "50bb78db-9822-412b-9b80-e568f909f091",
        "simulation_path": "Simulation

### Virtual BSDF Bench simulation

In [25]:
# Change the material property from mirror to bsdf type
opt_prop.set_surface_library()
opt_prop.sop_library.file_uri = assets_data_path / "R_test.anisotropicbsdf"
opt_prop.commit()
vbb = p.create_simulation(name="virtual_BSDF", feature_type=SimulationVirtualBSDF)
vbb.axis_system = [
    0.36,
    1.73,
    2.0,
    1.0,
    0.0,
    0.0,
    0.0,
    1.0,
    0.0,
    0.0,
    0.0,
    1.0,
]  # change the coordinate VBSDF to body center
vbb.commit()
results = vbb.compute_CPU()
print(results)

[upload_response {
  info {
    uri: "b8e589be-00fe-44b5-bf22-da8c0b6a6c3c"
    file_name: "virtual_BSDF.html"
    file_size: 27247
  }
}
, upload_response {
  info {
    uri: "5fed6974-87d6-4733-94f2-3be8829e23cd"
    file_name: "virtual_BSDF.anisotropicbsdf"
    file_size: 1261298
  }
}
]


## Simulation compute and stop

The compute_CPU, compute_GPU calls are blocking.
Thus it will return only once the simulation compute is finished.

If you want to have the possibility to stop it before it is finished, here an example.

In [26]:
from threading import Thread
from time import sleep

In [27]:
# Launch the compute_CPU in a thread and start the thread.
compute_thread = Thread(target=vbb.compute_CPU)
compute_thread.start()
# Wait 2 seconds then stop_computation
sleep(2)
vbb.stop_computation()
# Join the compute_thread
compute_thread.join()

In [28]:
speos.close()

True